In [ ]:
import os
import mne
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy
import re

from collections import defaultdict
from functions import preprocessing, utils, faster

The following notebook will:
- load the "block*" files
- concatenate blocks
- epoch the concatenated task blocks 
- reject artefact epochs
- plot epoch level TFR for quick visual

# 1. Load task blocks #

In [ ]:
# Session to preprocess
# session_id = "sub011 DBS OFF mSST"
session_id = "C009 mSST"
sub = session_id.split(' ') [0]

if "C" in session_id:
    condition = "noDBS"
else:
    condition = session_id.split(' ') [1] + ' ' + session_id.split(' ') [2]

saving_path = "C:\\Users\\Juliette\\Research\\Projects\\analysis_mSST\\results\\eeg_output\\single_sub"
onedrive_path = utils._get_onedrive_path()

data_path = utils._get_onedrive_path()
block_path = os.path.join(saving_path, session_id, "data")

# Automatically check how many blocks were extracted and create a list of blocks accordingly
numberof_block_files=0
for file in os.listdir(block_path):
        if file.endswith("postICA_EEGdata_eeg.fif"):
            numberof_block_files=numberof_block_files+1

if numberof_block_files==0:
    raise ValueError("No block files found in the specified directory.")
elif numberof_block_files==3:
    blocks = ["block1", "block2", "block3"]
elif numberof_block_files==4:
    blocks = ["block1", "block2", "block3", "block4"]

raw_blocks = {}  # dictionary to store blocks

for block_num in blocks:
    file_name = f"{sub}_{condition}_{block_num}_postICA_EEGdata_eeg.fif"
    file_path = os.path.join(block_path, file_name)
    raw_blocks[block_num] = mne.io.read_raw_fif(file_path, preload=True)
    ch_names = raw_blocks[block_num].info['ch_names']

In [ ]:
# Check that the correct files have been loaded
raw_blocks

# 2. Epoch each block separately, homogenize event mapping and concatenate #

In [ ]:
if "C" in session_id:     
    mSST_raw_behav_session_data_path = os.path.join(
            onedrive_path, sub, "raw_data", 'BEHAVIOR', 
            'mSST'
            )
else:
    mSST_raw_behav_session_data_path = os.path.join(
            onedrive_path, sub, "raw_data", 'BEHAVIOR', condition, 
            'mSST'
            )     
    
for filename in os.listdir(mSST_raw_behav_session_data_path):
        if filename.endswith(".csv"):
            fname = filename
filepath_behav = os.path.join(mSST_raw_behav_session_data_path, fname)
df = pd.read_csv(filepath_behav)

# return the index of the first row which is not filled by a Nan value:
start_task_index = df['blocks.thisRepN'].first_valid_index()
stop_task_index = df['blocks.thisRepN'].last_valid_index()
df_maintask = df.iloc[start_task_index:stop_task_index + 1]

# remove all useless columns to clean up dataframe
column_names = df_maintask.columns
columns_to_keep = [i for i in [
    'blocks.thisN', 'trial_loop.thisN', 'trial_type', 
    'continue_signal_time', 'stop_signal_time', 
    'fixation_cross.started', 'go_rectangle.started',
    'key_resp_experiment.keys', 'key_resp_experiment.corr', 'key_resp_experiment.rt',
    'early_press_resp.keys', 'early_press_resp.rt', 'early_press_resp.corr',
    'late_key_resp1.keys', 'late_key_resp1.rt', 
    'late_key_resp2.keys', 'late_key_resp2.rt'
    ] if i in column_names]

mini_df_maintask = df_maintask[columns_to_keep]
epochs_dict = {} # dictionnary to store epochs of each block

for block_num in blocks:
    print(f"Processing {block_num}...")
    # only keep trials of the block we are analyzing
    block_num_int = int(block_num.replace('block', ''))
    print(block_num_int)
    mini_df_maintask_block = mini_df_maintask[mini_df_maintask['blocks.thisN'] == (block_num_int - 1)].reset_index(drop=True)
    print(mini_df_maintask_block.shape)

    # remove the trials with early presses, as in these trials the cues were not presented (for mSST)
    early_presses = mini_df_maintask_block[mini_df_maintask_block['early_press_resp.corr'] == 1]
    early_presses_trials = list(early_presses.index)
    number_early_presses = len(early_presses_trials)

    # remove trials with early presses from the dataframe:
    mini_df_maintask_block_copy = mini_df_maintask_block.drop(early_presses_trials).reset_index(drop=True)

    # First generate global epochs (without taking into account success outcome)
    # events and event_id used for epochs creation
    raw = raw_blocks[block_num]
    events, event_id = mne.events_from_annotations(raw)
    epochs, filtered_event_dict = preprocessing.create_epochs(
            raw, 
            sub, 
            keys_to_keep = ['GC', 'GF', 'GO', 'GS', 'continue', 'stop'],
            tmin = -3.5,
            tmax = 3.5,
            baseline=None
            )
    n_epochs = len(epochs)

    # inverse mapping (event code -> label)
    inv_event_id = {v: k for k, v in event_id.items()}

    metadata = pd.DataFrame(index=np.arange(len(epochs)))
    metadata["event"] = [inv_event_id[e] for e in epochs.events[:, 2]]
    metadata["sample"] = epochs.events[:, 0]
    metadata["event_timing"] = epochs.events[:, 0] / raw.info['sfreq']  # in seconds
    metadata["trial_type"] = np.nan

    # LFP -> behavioral naming mapping
    mapping = {
        "GC": "go_continue_trial",
        "GO": "go_trial",
        "GF": "go_fast_trial",
        "GS": "stop_trial",
    }

    trial_mask = metadata["event"].isin(mapping.keys())

    assert trial_mask.sum() == len(mini_df_maintask_block_copy), \
        f"Mismatch: {trial_mask.sum()} LFP trials vs {len(mini_df_maintask_block_copy)} behavioral trials"

    # fill directly from behavioral file
    for col in mini_df_maintask_block_copy.columns:
        metadata.loc[trial_mask, col] = mini_df_maintask_block_copy[col].values

    for i in metadata.index:
        if metadata.loc[i, "event"] == "continue":
            # find the last GC before this
            prev_idx = metadata.loc[:i-1][metadata["event"] == "GC"].index[-1]
            metadata.loc[i, mini_df_maintask_block_copy.columns] = metadata.loc[prev_idx, mini_df_maintask_block_copy.columns]

        elif metadata.loc[i, "event"] == "stop":
            # find the last GS before this
            prev_idx = metadata.loc[:i-1][metadata["event"] == "GS"].index[-1]
            metadata.loc[i, mini_df_maintask_block_copy.columns] = metadata.loc[prev_idx, mini_df_maintask_block_copy.columns]

    epochs.metadata = metadata
    epochs_dict[block_num] = epochs


# Define the master mapping (keep keys consistent across blocks to facilitate concatenation)
master_event_id = {
    'GC': 1,
    'GF': 2,
    'GO': 3,
    'GS': 4,
    'continue': 5,
    'stop': 6
}

# Create a reverse mapping (code -> name) for easy lookup
inv_master_event_id = {v: k for k, v in master_event_id.items()}

for block_name, epochs in epochs_dict.items():
    print(f"Processing {block_name}...")
    
    # 1. Create a mapping from old event codes to new event names
    # We need to map the *old* event codes (in epochs.events[:, 2]) to the *new* codes
    # First, get the current event_id of this block
    current_event_id = epochs.event_id
    
    # 2. Build a translation map: old_code -> new_code
    # We need to ensure the names match the master_event_id keys
    translation_map = {}
    for name, old_code in current_event_id.items():
        if name in master_event_id:
            translation_map[old_code] = master_event_id[name]
        else:
            # Handle any unexpected events (optional: raise error or skip)
            print(f"Warning: Event '{name}' in {block_name} not in master_event_id. Skipping.")
            continue

    # 3. Update the events array
    # epochs.events[:, 2] contains the old codes. Replace them with new codes.
    old_codes = epochs.events[:, 2]
    new_codes = np.array([translation_map.get(code, 0) for code in old_codes]) # 0 for unknown (or handle error)
    
    # Update the events array in place
    epochs.events[:, 2] = new_codes
    
    # 4. Update the event_id dictionary of the Epochs object
    epochs.event_id = master_event_id.copy()

# Now all blocks have the same event_id mapping
# Concatenate
epochs_all = mne.concatenate_epochs([epochs_dict['block1'], epochs_dict['block2'], epochs_dict['block3']])

print(f"Concatenation successful! Total epochs: {len(epochs_all)}")

# 3. Reject "bad" epochs # 

## 3.1. OPTIONAL: Automatic method: FASTER algorithm ##
Use to get a first idea of which epochs are "bads"

In [ ]:
# Mark bad epochs using FASTER algorithm
cleaned_epochs = epochs_all.copy()
bad_epochs = faster.find_bad_epochs(cleaned_epochs, return_by_metric=False)
bad_epochs.sort()

bad_epochs

## 3.2. Manual epoch rejection ##

In [ ]:
%matplotlib qt

cleaned_epochs = epochs_all.copy()
ch_names = epochs_all.info['ch_names']
cleaned_epochs.plot(n_epochs=4, n_channels=len(ch_names), events=True, block=True);

In [ ]:
cleaned_epochs

# 4. Save cleaned epochs #

In [ ]:
epoch_save_path = os.path.join(saving_path, sub, condition, "epochs", "data")
os.makedirs(epoch_save_path, exist_ok=True)  # Create the directory if it doesn't exist

fig_save_path = os.path.join(saving_path, sub, condition, "epochs", "figures")
os.makedirs(fig_save_path, exist_ok=True)  # Create the directory if it doesn't exist


metadata_df = pd.DataFrame(cleaned_epochs.metadata)
# save both to csv (easier for later python import), and xlsx (easier to read in excel)
metadata_df.to_csv(os.path.join(epoch_save_path, f"{session_id}_cleaned-long-epo_metadata.csv"), index=True)
metadata_df.to_excel(os.path.join(epoch_save_path, f"{session_id}_cleaned-long-epo_metadata.xlsx"), index=True)

file_epoch = os.path.join(epoch_save_path, f"{session_id}_EEG_cleaned-long-epo.fif")
cleaned_epochs.save(file_epoch, overwrite=True)

# 5. OPTIONAL: Check epoch quality by plotting the TFR for specific trial types, e.g., ['GO_successful', 'GF_successful', 'GC_successful', 'GS_successful', 'GS_unsuccessful'] #

In [ ]:
######################
### TFR PARAMETERS ###
######################

decim = 1 
freqs = np.arange(1, 80, 1) 
# For 500ms time resolution at 1 Hz: n_cycles = 1 * 0.5 = 0.5
# For 50ms time resolution at 40 Hz: n_cycles = 40 * 0.05 = 2
# Linear interpolation between these points
#n_cycles = 0.5 + (freqs - 1) * (2 - 0.5) / (40 - 1)
#n_cycles = freqs / 2.0
n_cycles = np.minimum(np.maximum(freqs / 2.0, 2), 10)

tfr_args = dict(
    method="morlet",
    freqs=freqs,
    n_cycles=n_cycles,
    decim=decim,
    return_itc=False,
    average=False
)        

tmin_tmax = [-500, 1500]
vmin_vmax = [-70, 70]

In [ ]:
%matplotlib inline

for epoch_cond in ['GO_successful', 'GF_successful', 'GC_successful', 'GS_successful', 'GS_unsuccessful']:
    ch_interest = "Cz"

    t_min_max = [-500, 1500]

    epoch_type = epoch_cond.split('_')[0]
    outcome_str = epoch_cond.split('_')[1]

    outcome = 1.0 if outcome_str == 'successful' else 0.0

    type_mask = cleaned_epochs.metadata["event"] == epoch_type
    outcome_mask = cleaned_epochs.metadata["key_resp_experiment.corr"] == outcome
    data = cleaned_epochs[type_mask & outcome_mask]   

    channel_epochs = data.copy().pick([ch_interest])
    power_channel = channel_epochs.compute_tfr(**tfr_args)
    mean_power_channel = np.nanmean(power_channel.data, axis=0).squeeze()

    times = power_channel.times * 1000  # Convert to milliseconds
    freqs = power_channel.freqs

    baseline_indices = (times >= -500) & (times <= -200)
    baseline_power_channel = np.nanmean(mean_power_channel[:, baseline_indices], axis=1, keepdims=True)
    percentage_change_channel = (mean_power_channel - baseline_power_channel) / baseline_power_channel * 100

    time_indices = np.logical_and(times >= t_min_max[0], times <= t_min_max[1])
    sliced_data = percentage_change_channel[:, time_indices].squeeze()    

    plt.imshow(sliced_data, aspect='auto', origin='lower', 
            extent=[t_min_max[0], t_min_max[1], 
            tfr_args["freqs"][0], tfr_args["freqs"][-1]], 
            cmap='jet', vmin=vmin_vmax[0], vmax=vmin_vmax[-1]
    )
    plt.axvline(x=0, color='k', linestyle='--', linewidth=1)
    plt.title(f"Percentage Change in Power for {epoch_cond} - {ch_interest}")
    plt.xlabel("Time from GO cue (ms)")
    plt.ylabel("Frequency (Hz)")
    plt.colorbar(label="Percentage Change (%)")
    plt.tight_layout()
    plt.savefig(os.path.join(fig_save_path, f"{session_id}_{epoch_cond}_{ch_interest}_tfr.png"))
    plt.show()